# wflow SBM vs VAMPS — with specified initial groundwater level

This notebook is a variant of `sbm_vs_richards.ipynb` that starts both models with a
**specified initial water table depth** (`GW_DEPTH_MM`).  Change the single parameter
cell below and re-run all cells.

Both models evolve freely from the initial state:
- **VAMPS** uses `bottom = 4` (fixed head BC) with a hydrostatic-equilibrium theta
  profile from `GW_DEPTH_MM`.  The prescribed head at the bottom node keeps the water
  table effectively anchored at its initial depth.
- **SBM** initialises the saturated zone from `GW_DEPTH_MM` and updates the water
  table each timestep via drainable-porosity accounting.

| Setting | Effect |
|---|---|
| `GW_DEPTH_MM < SOILTHICKNESS` | Saturated zone within profile at t=0 |
| `GW_DEPTH_MM >= SOILTHICKNESS` | No initial saturation (free drainage) |

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')
from vampspy import Model

FIJI_DIR = os.path.join('..', 'examples', 'fiji')

## ▶ Parameter: groundwater depth

Change `GW_DEPTH_MM` here to set the water table for both models.

In [ ]:
# ---------------------------------------------------------------------------
# *** SET INITIAL WATER TABLE DEPTH HERE ***
# ---------------------------------------------------------------------------
GW_DEPTH_MM  = 1000.0    # mm below surface  (1000 mm = 1 m)
# ---------------------------------------------------------------------------

GW_DEPTH_CM  = GW_DEPTH_MM / 10.0
SOILTHICKNESS = 1540.0   # mm (77 layers x 20 mm)

print(f"GW depth (initial): {GW_DEPTH_MM:.0f} mm  ({GW_DEPTH_CM:.1f} cm)")
if GW_DEPTH_MM < SOILTHICKNESS:
    sat_mm = SOILTHICKNESS - GW_DEPTH_MM
    print(f"Saturated zone    : {sat_mm:.0f} mm at t=0")
else:
    print("Saturated zone    : none (GW below soil profile)")

## 1. SBM core functions

In [ ]:
def unsatzone_flow_layer(usd, kv_z, l_sat, c):
    if usd <= 0.0:
        return 0.0, 0.0
    st_sat   = max(0.0, usd - l_sat)
    st       = kv_z * min((usd / l_sat) ** c, 1.0)
    sum_ast  = min(st, st_sat)
    usd     -= sum_ast
    remainder = min(st - sum_ast, usd)
    its = max(1, int(np.ceil(remainder / 0.2)))
    for _ in range(its):
        st_i    = (kv_z / its) * min((usd / l_sat) ** c, 1.0)
        ast     = min(st_i, usd)
        usd    -= ast
        sum_ast += ast
    return usd, sum_ast


def vwc_brooks_corey(h, hb, theta_s, theta_r, c):
    if h < hb:
        par_lambda = 2.0 / (c - 3.0)
        return (theta_s - theta_r) * (hb / h) ** par_lambda + theta_r
    return theta_s


def head_brooks_corey(vwc, theta_s, theta_r, c, hb):
    par_lambda = 2.0 / (c - 3.0)
    if par_lambda > 0:
        return hb / ((vwc / (theta_s - theta_r)) ** (1.0 / par_lambda))
    return hb


def rwu_reduction_feddes(h, h1, h2, h3, h4, alpha_h1=0.0):
    if h < h4:  return 0.0
    if h < h3:  return (h - h4) / (h3 - h4)
    if alpha_h1 == 0.0:
        if h < h2:  return 1.0
        if h < h1:  return (h1 - h) / (h1 - h2)
        return 0.0
    return 1.0


def set_layerthickness(zi, cum_depth, act_thick):
    """Unsaturated thickness of each layer given water table depth zi [mm]."""
    out = np.zeros_like(act_thick)
    for i, (z0, z1, dz) in enumerate(zip(cum_depth[:-1], cum_depth[1:], act_thick)):
        if zi >= z1:
            out[i] = dz
        elif zi > z0:
            out[i] = zi - z0
        # else: fully saturated → 0
    return out

## 2. Soil parameters

In [ ]:
LAYER_BOUNDS = np.array([0.0, 280.0, 720.0, 1540.0])   # mm
ACT_THICK    = np.diff(LAYER_BOUNDS)                     # [280, 440, 820] mm
CUM_DEPTH    = LAYER_BOUNDS
NLAYERS      = len(ACT_THICK)

THETA_S   = np.array([0.60,  0.64,  0.60 ])
THETA_R   = np.array([0.08,  0.08,  0.08 ])
LAMBDA_BC = np.array([0.098, 0.094, 0.094])
C_BC      = 3.0 + 2.0 / LAMBDA_BC
HB_LAYER  = np.array([-16.4, -23.8, -23.8])   # cm
KV_LAYER  = np.array([1800.0, 380.0, 3.0]) * 10.0   # mm d-1
KVFRAC    = np.ones(NLAYERS)

SOILWATERCAPACITY = (THETA_S - THETA_R) * ACT_THICK
MAXLEAKAGE = KV_LAYER[-1]   # 30 mm d-1

H1 = -5.0; H2 = -50.0; H3_HIGH = -800.0; H3_LOW = -1000.0
H4 = -12000.0; ALPHA_H1 = 0.0
ROOTINGDEPTH  = 1200.0   # mm
INFILTCAPSOIL = 200.0    # mm d-1

def field_capacity_bc(h_fc=-100.0):
    fc = (THETA_S - THETA_R) * (HB_LAYER / h_fc) ** LAMBDA_BC + THETA_R
    return np.minimum(fc, THETA_S)

THETA_FC       = field_capacity_bc()
THETA_DRAINABLE = np.maximum(THETA_S - THETA_FC, 0.02)

print('Layer thicknesses (mm) :', ACT_THICK)
print('Brooks-Corey c         :', C_BC.round(2))
print('Drainable porosity     :', THETA_DRAINABLE.round(3))

## 3. Saturated zone helpers

In [ ]:
def satwaterdepth_from_zi(zi):
    """Drainable water in the saturated zone [mm] for water table at zi [mm]."""
    swd = 0.0
    for k in range(NLAYERS):
        z0, z1 = CUM_DEPTH[k], CUM_DEPTH[k + 1]
        sat_thick = max(0.0, z1 - max(zi, z0))
        swd += sat_thick * THETA_DRAINABLE[k]
    return swd


def zi_from_satwaterdepth(swd):
    """Water table depth [mm] from drainable saturated storage [mm]."""
    remaining = max(0.0, swd)
    zi = SOILTHICKNESS
    for k in range(NLAYERS - 1, -1, -1):
        layer_drain = ACT_THICK[k] * THETA_DRAINABLE[k]
        if remaining >= layer_drain:
            remaining -= layer_drain
            zi = CUM_DEPTH[k]
        else:
            zi = CUM_DEPTH[k + 1] - remaining / THETA_DRAINABLE[k]
            remaining = 0.0
            break
    return max(0.0, zi)


# Verify round-trip
swd_test = satwaterdepth_from_zi(GW_DEPTH_MM)
zi_test  = zi_from_satwaterdepth(swd_test)
print(f'GW_DEPTH_MM = {GW_DEPTH_MM:.1f}  →  satwaterdepth = {swd_test:.2f} mm  '
      f'→  zi_roundtrip = {zi_test:.2f} mm')

## 4. Initial conditions

Above the water table: original Fiji theta profile.  
Below the water table: theta_s (fully saturated).

In [ ]:
# Brooks-Corey retention curve (same parameters as VAMPS config)
BC_PARAMS = [
    # (theta_s, theta_r, lambda, hb_cm, z_range_mm)
    (0.60, 0.08, 0.098, -16.4,  (0,   280)),
    (0.64, 0.08, 0.094, -23.8,  (280, 720)),
    (0.60, 0.08, 0.094, -23.8,  (720, 1540)),
]

def theta_equilibrium_bc(z_mm, gw_depth_mm):
    """Equilibrium theta at depth z_mm for water table at gw_depth_mm."""    h_cm = (gw_depth_mm - z_mm) / 10.0   # pressure head [cm]; negative above WT
    for (ts, tr, lam, hb, (z0, z1)) in BC_PARAMS:
        if z0 <= z_mm < z1:
            if h_cm >= 0.0:
                return ts
            if h_cm > hb:   # wetter than air entry but unsaturated (|h| < |hb|)
                return ts
            return (ts - tr) * (hb / h_cm) ** lam + tr
    return BC_PARAMS[-1][0]   # fallback: theta_s of last layer

N_VAMPS    = 77
DZ_VAMPS_MM = 20.0

# Equilibrium theta profile for VAMPS
theta_initial_vamps = np.array([
    theta_equilibrium_bc((i + 0.5) * DZ_VAMPS_MM, GW_DEPTH_MM)
    for i in range(N_VAMPS)
])

# SBM initial conditions
idx = [0, 14, 36, 77]
theta_ini_sbm = np.array([
    theta_initial_vamps[idx[k]:idx[k+1]].mean() for k in range(NLAYERS)
])
unsat_thick_ini       = set_layerthickness(GW_DEPTH_MM, CUM_DEPTH, ACT_THICK)
ustorelayerdepth_ini  = np.array([
    max(0.0, (theta_ini_sbm[k] - THETA_R[k]) * unsat_thick_ini[k])
    for k in range(NLAYERS)
])
satwaterdepth_ini = satwaterdepth_from_zi(GW_DEPTH_MM)
zi_ini            = GW_DEPTH_MM

total_water_vamps = (theta_initial_vamps * DZ_VAMPS_MM).sum()
total_water_sbm   = (ustorelayerdepth_ini.sum()
                     + THETA_R.mean() * SOILTHICKNESS
                     + satwaterdepth_ini)

print(f"Initial GW depth          : {zi_ini:.0f} mm")
print(f"Saturated water depth     : {satwaterdepth_ini:.1f} mm")
print(f"Unsat storage per layer   : {ustorelayerdepth_ini.round(1)}")
print(f"Total soil water (VAMPS)  : {total_water_vamps:.1f} mm")
print(f"Total soil water (SBM)    : {total_water_sbm:.1f} mm")

## 5. Forcing

In [ ]:
BASE_STEPS = 61
NSTEPS     = 180

def load_prn(fname):
    return np.loadtxt(os.path.join(FIJI_DIR, fname))[:BASE_STEPS, 1]

precip_cm = load_prn('precip.prn')
rnet_W    = load_prn('rnet.prn')
temp_C    = load_prn('newt.prn')
rh        = load_prn('rh.prn')
wind      = load_prn('wind.prn')

precip_mm = precip_cm * 10.0

def tile_to(arr, n):
    return np.tile(arr, -(-n // len(arr)))[:n]

precip_mm = tile_to(precip_mm, NSTEPS)
rnet_W    = tile_to(rnet_W,    NSTEPS)
temp_C    = tile_to(temp_C,    NSTEPS)
rh        = tile_to(rh,        NSTEPS)
wind      = tile_to(wind,      NSTEPS)

RA_S, RS_S  = 7.0, 60.0
lam_MJ      = 2.45
gamma_kPa   = 0.067
rho_a_cp    = 1.2 * 1.013e-3

esat   = 0.6108 * np.exp(17.27 * temp_C / (temp_C + 237.3))
vpd    = esat * (1.0 - rh / 100.0)
Delta  = 4098 * esat / (temp_C + 237.3) ** 2
Rn_MJ  = rnet_W * 86400 / 1e6
ra_d   = RA_S / 86400

ET_PM = np.maximum(
    (Delta * Rn_MJ + rho_a_cp * vpd / ra_d)
    / (lam_MJ * (Delta + gamma_kPa * (1.0 + RS_S / RA_S))),
    0.0,
)

RNET_ABSORB           = 0.975
pot_transpiration     = ET_PM * RNET_ABSORB
pot_soilevaporation   = ET_PM * (1.0 - RNET_ABSORB)
INTERCEPTION_FRAC     = 0.147
interception          = precip_mm * INTERCEPTION_FRAC
pot_infiltration      = np.maximum(precip_mm - interception, 0.0)

print(f'Timesteps           : {NSTEPS}')
print(f'Mean precip  mm/d   : {precip_mm.mean():.2f}')
print(f'Mean ET_PM   mm/d   : {ET_PM.mean():.2f}')
print(f'Mean pot T   mm/d   : {pot_transpiration.mean():.2f}')

## 6. SBM model run — with saturated zone

Water table (`zi`) is updated every timestep from the saturated zone storage.
Unsaturated layer thicknesses are recomputed accordingly.  
Capillary rise = 0 (conservative tropical forest assumption).

In [ ]:
def run_sbm_gw(nsteps, pot_infiltration, pot_transpiration, pot_soilevaporation):
    usd          = ustorelayerdepth_ini.copy()
    zi           = float(zi_ini)
    satwaterdepth = float(satwaterdepth_ini)

    keys = ['ustoredepth', 'satwaterdepth', 'zi', 'deep_perc',
            'infiltration', 'transpiration', 'soilevap', 'runoff',
            'total_soilwater']
    out = {k: np.zeros(nsteps) for k in keys}
    out['usd'] = np.zeros((nsteps, NLAYERS))
    out['vwc'] = np.zeros((nsteps, NLAYERS))

    def h3_from_tp(tp):
        if tp <= 1.0: return H3_LOW
        if tp <  5.0: return H3_LOW + (H3_HIGH - H3_LOW) * (tp - 1.0) / 4.0
        return H3_HIGH

    rootfrac = np.zeros(NLAYERS)
    for k in range(NLAYERS):
        z0, z1 = CUM_DEPTH[k], CUM_DEPTH[k + 1]
        if ROOTINGDEPTH >= z1:
            rootfrac[k] = ACT_THICK[k] / ROOTINGDEPTH
        else:
            rootfrac[k] = max(ROOTINGDEPTH - z0, 0.0) / ROOTINGDEPTH

    for step in range(nsteps):
        h3 = h3_from_tp(pot_transpiration[step])

        # Unsaturated thickness of each layer given current water table
        unsat_thick = set_layerthickness(zi, CUM_DEPTH, ACT_THICK)

        # --- Infiltration ---
        water_in    = pot_infiltration[step]
        l_sat_all   = unsat_thick * (THETA_S - THETA_R)
        ustorecap   = max(0.0, (l_sat_all - usd).sum())
        infil       = min(min(INFILTCAPSOIL, water_in), ustorecap)
        infiltexcess = water_in - infil

        # --- Unsaturated zone flow layer by layer ---
        flow_in = infil
        for m in range(NLAYERS):
            if unsat_thick[m] <= 0.0:
                # fully saturated layer: pass all water to recharge
                satwaterdepth += flow_in
                flow_in = 0.0
                break
            l_sat_m = unsat_thick[m] * (THETA_S[m] - THETA_R[m])
            kv_z    = KV_LAYER[m] * KVFRAC[m]
            usd_m   = usd[m] + flow_in
            usd[m], flow_in = unsatzone_flow_layer(usd_m, kv_z, l_sat_m, C_BC[m])
        else:
            # drainage from bottom unsaturated layer → recharge
            satwaterdepth += flow_in

        # --- Soil evaporation (top layer) ---
        l_sat_0 = max(unsat_thick[0], 1e-6) * (THETA_S[0] - THETA_R[0])
        soilevap = pot_soilevaporation[step] * min(1.0, usd[0] / l_sat_0) if l_sat_0 > 0 else 0.0
        soilevap = min(soilevap, usd[0])
        usd[0]  -= soilevap

        # --- Transpiration (Feddes, unsaturated layers only) ---
        sum_rf = rootfrac[:NLAYERS].sum()
        scale  = max(1.0 / sum_rf, 1.0) if sum_rf > 0 else 0.0
        actevapustore = 0.0
        for k in range(NLAYERS):
            thick_k = unsat_thick[k]
            if thick_k <= 0: continue
            vwc_k   = max(usd[k] / thick_k, 1e-7)
            h_k     = head_brooks_corey(vwc_k, THETA_S[k], THETA_R[k], C_BC[k], HB_LAYER[k])
            alpha_k = rwu_reduction_feddes(h_k, H1, H2, h3, H4, ALPHA_H1)
            availcap = min(1.0, max(0.0, (ROOTINGDEPTH - CUM_DEPTH[k]) / thick_k))
            maxextr  = usd[k] * availcap
            rf_sc    = scale * rootfrac[k] if ROOTINGDEPTH > 0 else 0.0
            evap_k   = min(alpha_k * rf_sc * pot_transpiration[step], maxextr)
            usd[k]  -= evap_k
            actevapustore += evap_k

        # --- Overflow check (bottom → top) ---
        ustoredepth_excess = 0.0
        for k in range(NLAYERS - 1, -1, -1):
            l_sat_k = unsat_thick[k] * (THETA_S[k] - THETA_R[k])
            exc = max(0.0, usd[k] - l_sat_k)
            usd[k] -= exc
            if k > 0:
                usd[k - 1] += exc
            else:
                ustoredepth_excess = exc
        runoff = max(0.0, infiltexcess + ustoredepth_excess)

        # --- Saturated zone: leakage at bottom ---
        leakage = min(MAXLEAKAGE, satwaterdepth)
        satwaterdepth -= leakage
        satwaterdepth  = max(0.0, satwaterdepth)

        # --- Update water table ---
        zi = zi_from_satwaterdepth(satwaterdepth)

        # --- VWC per layer ---
        vwc = np.where(
            unsat_thick > 0,
            np.maximum(usd / np.maximum(unsat_thick, 1e-6) + THETA_R, THETA_R),
            THETA_S
        )

        total_sw = (usd.sum()
                    + THETA_R.mean() * SOILTHICKNESS
                    + satwaterdepth)

        out['ustoredepth'][step]    = usd.sum()
        out['satwaterdepth'][step]  = satwaterdepth
        out['zi'][step]             = zi
        out['deep_perc'][step]      = leakage
        out['infiltration'][step]   = infil
        out['transpiration'][step]  = actevapustore
        out['soilevap'][step]       = soilevap
        out['runoff'][step]         = runoff
        out['total_soilwater'][step] = total_sw
        out['usd'][step]            = usd.copy()
        out['vwc'][step]            = vwc

    return out


sbm = run_sbm_gw(NSTEPS, pot_infiltration, pot_transpiration, pot_soilevaporation)
print('SBM run complete.')
print(f'  Initial zi : {zi_ini:.0f} mm')
print(f'  Final   zi : {sbm["zi"][-1]:.0f} mm')
print(f'  Mean actual T : {sbm["transpiration"].mean():.3f} mm/d')
print(f'  Total leakage : {sbm["deep_perc"].sum():.1f} mm')

## 7. VAMPS run — fixed head BC, hydrostatic initial profile

Uses `bottom = 4` (fixed head) with `initprof = 2` (hydrostatic from `gw_initial`).
The prescribed head at the bottom node is `h_bot = bottom_depth - GW_DEPTH_CM`, which
keeps the GW level consistent with the initial profile and avoids solver convergence
issues from a discontinuous head gradient.

In [ ]:
prn_times = np.arange(1, NSTEPS + 1, dtype=float)

def write_prn(path, times, values_mm_per_day):
    values_cm = values_mm_per_day / 10.0
    with open(path, 'w') as f:
        for t, v in zip(times, values_cm):
            f.write(f'{t:.2f}\t{v:.8f}\n')

write_prn(os.path.join(FIJI_DIR, 'precip_tiled.prn'), prn_times, precip_mm)
write_prn(os.path.join(FIJI_DIR, 'ptr.prn'),          prn_times, pot_transpiration)
write_prn(os.path.join(FIJI_DIR, 'spe.prn'),          prn_times, pot_soilevaporation)
write_prn(os.path.join(FIJI_DIR, 'inr.prn'),          prn_times, interception)

# Fixed head at bottom node: pressure head = node_depth - GW_DEPTH_CM
# Bottom node centre is at (layers - 0.5) * layer_thickness_cm
_layers    = 77
_dz_cm     = 2.0
_bot_depth = (_layers - 0.5) * _dz_cm          # cm, depth of bottom node centre
_hea_cm    = _bot_depth - GW_DEPTH_CM           # cm, positive when node is below GW

_hea_path = os.path.join(FIJI_DIR, 'hea_gw.prn')
with open(_hea_path, 'w') as _f:
    for _t in prn_times:
        _f.write(f'{_t:.2f}\t{_hea_cm:.4f}\n')

vamps_cfg = f"""\
[vamps]
iniinmem=1

[run]
outputfile = fiji_bandC_gw.out

[determine]
soilmoisture = 1

[top]
system = 4

[time]
steps = {NSTEPS}

[ts]
pre = precip_tiled.prn
ptr = ptr.prn
spe = spe.prn
inr = inr.prn
hea = hea_gw.prn

[roots]
swsink = 1
swhypr = 0
swupfu = 0
depth  = 120.0
hlim1  = -5.0
hlim2u = -50.0
hlim2l = -50.0
hlim3h = -800.0
hlim3l = -1000.0
hlim3  = -1800.0
hlim4  = -12000.0

[soil]
dtmin   = 0.1E-3
dtmax   = 0.5E-2
mbck    = 0
swredu  = 1
smooth  = 5
cofred  = 0.35
outdir  = output
pondmx  = 0.0
verbose = 1
layers  = 77
bottom  = 4
initprof = 2
gw_initial = {GW_DEPTH_CM:.2f}
mktable  = 1
dumptables = 1

[layer_0]
description  = Tulasewa top layer
thickness    = 2.000000
soilsection  = st_0

[st_0]
method         = 6
thetas         = 0.6
theta_residual = 0.08
lambda         = 0.098
hb             = -16.4
ksat           = 1800

[layer_14]
description  = Tulasewa 30-75 cm layer
thickness    = 2.000000
soilsection  = st_1

[st_1]
method         = 6
thetas         = 0.64
theta_residual = 0.08
lambda         = 0.094
hb             = -23.8
ksat           = 380.0

[layer_36]
description  = Tulsewa deep layer > 75 cm
thickness    = 2.000000
soilsection  = st_2

[st_2]
method         = 6
thetas         = 0.6
theta_residual = 0.08
lambda         = 0.094
hb             = -23.8
ksat           = 3.0
"""

_inp_path = os.path.join(FIJI_DIR, 'fiji_bandC_gw.inp')
with open(_inp_path, 'w') as _f:
    _f.write(vamps_cfg)
_orig_dir = os.getcwd()
os.chdir(FIJI_DIR)
_m = Model.from_file('fiji_bandC_gw.inp')
vamps_raw = _m.run_stepwise(firststep=1.0)
os.chdir(_orig_dir)

vamps = {k: np.array(v) for k, v in vamps_raw.items()
         if k in ['t', 'volact', 'SMD', 'qtop', 'qbot', 'avgtheta',
                  'transpiration', 'soilevaporation', 'interception',
                  'theta', 'h', 'k', 'q', 'qrot', 'masbal', 'gwl']}

vamps_actual_T = vamps['qrot'].sum(axis=1) * 10
vamps_actual_E = vamps['soilevaporation'] * 10
vamps_actual_I = vamps['interception']    * 10

def vamps_wtd(h_profile, dz_cm=2.0):
    """Water table depth [mm] from VAMPS h profile (h in cm, dz in cm)."""
    z_centers = (np.arange(len(h_profile)) + 0.5) * dz_cm
    below = np.where(h_profile >= 0)[0]
    if len(below) == 0:
        return z_centers[-1] * 10.0
    return z_centers[below[0]] * 10.0

vamps_zi = np.array([vamps_wtd(vamps['h'][t]) for t in range(NSTEPS)])

print(f'VAMPS run complete  (bottom=4, initprof=2, GW_DEPTH={GW_DEPTH_MM:.0f} mm, hea={_hea_cm:.1f} cm)')
print(f'  Mean actual T : {vamps_actual_T.mean():.3f} mm/d')
print(f'  Total |qbot|  : {abs(vamps["qbot"]).sum()*10:.1f} mm')
print(f'  Mean WTD      : {vamps_zi.mean():.0f} mm')

## 8. Comparison plots

In [ ]:
t = np.arange(1, NSTEPS + 1)
idx = [0, 14, 36, 77]  # VAMPS layer indices for SBM layer boundaries

fig = plt.figure(figsize=(14, 18))
gs  = gridspec.GridSpec(5, 2, figure=fig, hspace=0.50, wspace=0.35)

# (a) Total soil water
ax = fig.add_subplot(gs[0, 0])
ax.plot(t, vamps['volact'] * 10, 'b-',  lw=2, label='VAMPS')
ax.plot(t, sbm['total_soilwater'],   'r--', lw=2, label='SBM')
ax.set_title('(a) Total soil water'); ax.set_ylabel('mm')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (b) Water table depth
ax = fig.add_subplot(gs[0, 1])
ax.plot(t, vamps_zi,   'b-',  lw=2, label='VAMPS WTD')
ax.plot(t, sbm['zi'],  'r--', lw=2, label='SBM zi')
ax.axhline(SOILTHICKNESS, color='gray', ls=':', lw=1, label='Soil bottom')
ax.invert_yaxis()
ax.set_title('(b) Water table depth'); ax.set_ylabel('mm below surface')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (c) Transpiration
ax = fig.add_subplot(gs[1, 0])
ax.plot(t, pot_transpiration,   'k--', lw=1.5, label='Potential')
ax.plot(t, vamps_actual_T,      'b-',  lw=2,   label='VAMPS actual')
ax.plot(t, sbm['transpiration'], 'r--', lw=2,  label='SBM actual')
ax.set_title('(c) Transpiration'); ax.set_ylabel('mm d⁻¹')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (d) Soil evaporation
ax = fig.add_subplot(gs[1, 1])
ax.plot(t, vamps_actual_E,  'b-',  lw=2, label='VAMPS')
ax.plot(t, sbm['soilevap'], 'r--', lw=2, label='SBM')
ax.set_title('(d) Soil evaporation'); ax.set_ylabel('mm d⁻¹')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (e) Bottom drainage / leakage
ax = fig.add_subplot(gs[2, 0])
ax.plot(t, abs(vamps['qbot']) * 10, 'b-',  lw=2, label='VAMPS |qbot|')
ax.plot(t, sbm['deep_perc'],        'r--', lw=2, label='SBM leakage')
ax.set_title('(e) Bottom drainage / leakage'); ax.set_ylabel('mm d⁻¹')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (f) Surface runoff
ax = fig.add_subplot(gs[2, 1])
ax.bar(t, precip_mm, alpha=0.2, color='steelblue', label='Precip')
ax.plot(t, np.maximum(vamps['qtop'], 0) * 10, 'b-',  lw=2, label='VAMPS runoff')
ax.plot(t, sbm['runoff'],                      'r--', lw=2, label='SBM runoff')
ax.set_title('(f) Surface runoff'); ax.set_ylabel('mm d⁻¹')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (g) Mean theta per SBM layer
ax = fig.add_subplot(gs[3, :])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
labels = ['Top (0–280 mm)', 'Mid (280–720 mm)', 'Deep (720–1540 mm)']
for k in range(NLAYERS):
    v_mean = vamps['theta'][:, idx[k]:idx[k+1]].mean(axis=1)
    ax.plot(t, v_mean,            '-',  color=colors[k], lw=1.5, label=f'VAMPS {labels[k]}')
    ax.plot(t, sbm['vwc'][:, k],  '--', color=colors[k], lw=1.5, label=f'SBM {labels[k]}')
ax.set_title('(g) Mean θ per layer  (solid=VAMPS, dashed=SBM)')
ax.set_ylabel('θ'); ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# (h) Cumulative fluxes
ax = fig.add_subplot(gs[4, :])
ax.plot(t, np.cumsum(precip_mm),                          'k-',  lw=1.5, label='Precipitation')
ax.plot(t, np.cumsum(vamps_actual_T + vamps_actual_E),    'b-',  lw=2,   label='VAMPS ET')
ax.plot(t, np.cumsum(sbm['transpiration'] + sbm['soilevap']), 'r--', lw=2, label='SBM ET')
ax.plot(t, np.cumsum(abs(vamps['qbot']) * 10),            'b:',  lw=1.5, label='VAMPS bottom')
ax.plot(t, np.cumsum(sbm['deep_perc']),                   'r:',  lw=1.5, label='SBM leakage')
ax.set_title('(h) Cumulative fluxes'); ax.set_ylabel('mm')
ax.legend(fontsize=8, ncol=3); ax.grid(True, alpha=0.3)
ax.set_xlabel('Day')

fig.suptitle(
    f'wflow SBM vs VAMPS — GW depth = {GW_DEPTH_MM:.0f} mm  '
    f'(solid=VAMPS Richards, dashed=SBM Brooks-Corey)',
    fontsize=11, y=1.01
)
plt.savefig(f'sbm_vs_richards_gw{int(GW_DEPTH_MM)}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved sbm_vs_richards_gw{int(GW_DEPTH_MM)}.png')

## 9. θ profiles at selected timesteps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 7), sharey=True)
t_plot = [0, NSTEPS // 4, NSTEPS - 1]

for ax, ti in zip(axes, t_plot):
    z_mid = (np.arange(N_VAMPS) + 0.5) * DZ_VAMPS_MM
    ax.plot(vamps['theta'][ti], z_mid, 'b-', lw=2, label='VAMPS')

    # SBM block profile
    for k in range(NLAYERS):
        z0, z1 = CUM_DEPTH[k], CUM_DEPTH[k + 1]
        ax.fill_betweenx([z0, z1], sbm['vwc'][ti, k], alpha=0.25, color='red')
        ax.plot([sbm['vwc'][ti, k]] * 2, [z0, z1],
                'r--', lw=2, label='SBM' if k == 0 else '')

    # Water table lines
    ax.axhline(vamps_zi[ti], color='blue',  lw=1, ls='-.', alpha=0.7, label=f'VAMPS WTD')
    ax.axhline(sbm['zi'][ti], color='red',  lw=1, ls='-.', alpha=0.7, label=f'SBM WTD')

    ax.set_title(f'Day {ti + 1}\nWTD: V={vamps_zi[ti]:.0f} S={sbm["zi"][ti]:.0f} mm')
    ax.set_xlabel('θ'); ax.invert_yaxis()
    ax.set_xlim(0.0, 0.70)
    if ax is axes[0]:
        ax.set_ylabel('Depth (mm)')
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8)
fig.suptitle(
    f'θ profiles — GW depth = {GW_DEPTH_MM:.0f} mm  '
    f'(dash-dot = water table)',
    fontsize=10
)
plt.tight_layout()
plt.show()

## 10. Water balance summary

In [ ]:
total_precip = precip_mm.sum()

vamps_T  = vamps_actual_T.sum()
vamps_E  = vamps_actual_E.sum()
vamps_I  = vamps_actual_I.sum()
vamps_D  = abs(vamps['qbot'].sum() * 10)
vamps_dS = (vamps['volact'][-1] - vamps['volact'][0]) * 10

sbm_T    = sbm['transpiration'].sum()
sbm_E    = sbm['soilevap'].sum()
sbm_I    = interception.sum()
sbm_D    = sbm['deep_perc'].sum()
sbm_Q    = sbm['runoff'].sum()
sbm_dS   = sbm['total_soilwater'][-1] - sbm['total_soilwater'][0]

sbm_wbal = total_precip - (sbm_T + sbm_E + sbm_I) - sbm_Q - sbm_D - sbm_dS

print(f'GW depth: {GW_DEPTH_MM:.0f} mm')
print('=' * 65)
print(f'{"Component":<35} {"VAMPS":>8} {"SBM":>8}  units')
print('-' * 65)
print(f'{"Precipitation":<35} {total_precip:>8.1f} {total_precip:>8.1f}  mm')
print(f'{"Interception":<35} {vamps_I:>8.1f} {sbm_I:>8.1f}  mm')
print(f'{"Transpiration (actual)":<35} {vamps_T:>8.1f} {sbm_T:>8.1f}  mm')
print(f'{"Soil evaporation":<35} {vamps_E:>8.1f} {sbm_E:>8.1f}  mm')
print(f'{"Total ET (T+E+I)":<35} {vamps_T+vamps_E+vamps_I:>8.1f} {sbm_T+sbm_E+sbm_I:>8.1f}  mm')
print(f'{"Bottom drainage":<35} {vamps_D:>8.1f} {sbm_D:>8.1f}  mm')
print(f'{"Runoff":<35} {"n/a":>8} {sbm_Q:>8.1f}  mm')
print(f'{"Delta storage":<35} {vamps_dS:>8.1f} {sbm_dS:>8.1f}  mm')
print('=' * 65)
print(f'SBM balance residual: {sbm_wbal:+.2f} mm')
print(f'VAMPS masbal at end : {vamps["masbal"][-1]:.2f} cm')
print()
print(f'Initial WTD : {zi_ini:.0f} mm')
print(f'Final WTD   : VAMPS {vamps_zi[-1]:.0f} mm  |  SBM {sbm["zi"][-1]:.0f} mm')